# Notebook 6: Statistical Analysis and Figures

Reproduce the key analyses, tables, and figures from the paper:
1. Gender proportions over time, by journal rank, by country
2. Topic × gender analysis
3. Sex/gender content prevalence and categories
4. Spearman correlation: female authorship vs. gender content
5. Bayesian logistic regression
6. All key figures

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
import warnings
warnings.filterwarnings("ignore")

# Set plotting style
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150
os.makedirs("figures", exist_ok=True)

# Load all datasets
articles_all = pd.read_parquet("data/processed/articles_gendered.parquet")
articles_topics = pd.read_parquet("data/processed/articles_gender_content.parquet")
authorships = pd.read_parquet("data/processed/authorships_gendered.parquet")
journal_list = pd.read_csv("data/processed/journal_list.csv")

print(f"All articles: {len(articles_all):,}")
print(f"Articles with topics + gender content: {len(articles_topics):,}")
print(f"Authorships: {len(authorships):,}")

## 1. Overall Gender Proportions

Paper finding: 83.3% male, 16.7% female authorship

In [ ]:
# Overall gender breakdown at authorship level
classified = authorships[authorships["gender"].isin(["male", "female"])]
gender_counts = classified["gender"].value_counts()
gender_pcts = classified["gender"].value_counts(normalize=True) * 100

print("Overall Gender Distribution (authorship level):")
print(f"  Male:   {gender_counts['male']:,} ({gender_pcts['male']:.1f}%)")
print(f"  Female: {gender_counts['female']:,} ({gender_pcts['female']:.1f}%)")
print(f"  Unclassified: {(authorships['gender'] == 'unclassified').sum():,}")
print(f"\nPaper: Male 83.3%, Female 16.7%")

# Article-level stats
classified_articles = articles_all[articles_all["pct_female"].notna()]
print(f"\nArticle-level (mean % female across articles): {classified_articles['pct_female'].mean():.1%}")

# Distribution of female authorship proportion
print(f"\nArticles with >80% male authorship: {(classified_articles['pct_male'] > 0.8).mean():.1%}")
print(f"Articles with >80% female authorship: {(classified_articles['pct_female'] > 0.8).mean():.1%}")
print(f"Paper: 70.7% have >80% male, 4.3% have >80% female")

## 2. Female Authorship Over Time

In [ ]:
# Female authorship over time
yearly_gender = classified_articles.groupby("publication_year").agg(
    mean_pct_female=("pct_female", "mean"),
    n_articles=("pct_female", "count"),
).reset_index()

# Filter to years with sufficient data
yearly_gender = yearly_gender[yearly_gender["n_articles"] >= 10]

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(yearly_gender["publication_year"], yearly_gender["mean_pct_female"] * 100,
        color="steelblue", linewidth=2, label="% Female Authorship")

# Rolling average
if len(yearly_gender) > 5:
    rolling = yearly_gender.set_index("publication_year")["mean_pct_female"].rolling(5, center=True).mean() * 100
    ax.plot(rolling.index, rolling.values, color="darkred", linewidth=2.5,
            linestyle="--", label="5-year Rolling Average")

ax.set_xlabel("Publication Year")
ax.set_ylabel("Female Authorship (%)")
ax.set_title("Female Authorship in Academic Finance Over Time")
ax.legend()
ax.set_ylim(0, None)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figures/female_authorship_over_time.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Female Authorship by Journal Rank

Paper finding: Male dominance increases with journal prestige (77.3% in C journals → 85.6% in A* journals)

In [ ]:
# Female authorship by ABDC journal rank
rank_order = ["A*", "A", "B", "C"]
rank_gender = classified_articles.groupby("abdc_rank").agg(
    mean_pct_female=("pct_female", "mean"),
    mean_pct_male=("pct_male", "mean"),
    n_articles=("pct_female", "count"),
).reindex(rank_order)

print("Female Authorship by Journal Rank:")
for rank in rank_order:
    if rank in rank_gender.index:
        row = rank_gender.loc[rank]
        print(f"  {rank:3s}: {row['mean_pct_female']*100:.1f}% female, "
              f"{row['mean_pct_male']*100:.1f}% male (n={row['n_articles']:,.0f})")

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#d32f2f", "#f57c00", "#1976d2", "#388e3c"]
bars = ax.bar(rank_order, rank_gender["mean_pct_female"] * 100, color=colors, edgecolor="black", alpha=0.85)

# Add value labels
for bar, rank in zip(bars, rank_order):
    if rank in rank_gender.index:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{rank_gender.loc[rank, 'mean_pct_female']*100:.1f}%",
                ha="center", fontweight="bold")

ax.set_xlabel("ABDC Journal Rank")
ax.set_ylabel("Female Authorship (%)")
ax.set_title("Female Authorship by Journal Prestige")
ax.set_ylim(0, max(rank_gender["mean_pct_female"] * 100) * 1.2)
plt.tight_layout()
plt.savefig("figures/female_authorship_by_rank.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Female Authorship by Country

In [ ]:
# Gender by country (using first author's country)
first_authors = authorships[authorships["author_position"] == "first"].copy()
first_classified = first_authors[first_authors["gender"].isin(["male", "female"])]
first_classified = first_classified[first_classified["country_code"].notna()]

country_gender = first_classified.groupby("country_code").agg(
    n_authorships=("gender", "count"),
    n_female=("gender", lambda x: (x == "female").sum()),
).reset_index()
country_gender["pct_female"] = country_gender["n_female"] / country_gender["n_authorships"] * 100

# Filter to countries with at least 50 first authorships
country_gender = country_gender[country_gender["n_authorships"] >= 50].sort_values("pct_female", ascending=False)

# Top 20 countries by volume
top_countries = country_gender.nlargest(20, "n_authorships").sort_values("pct_female", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top_countries["country_code"], top_countries["pct_female"],
               color="steelblue", edgecolor="black", alpha=0.85)
ax.set_xlabel("Female First Authorship (%)")
ax.set_title("Female First Authorship by Country (Top 20 by Volume)")
ax.axvline(x=50, color="red", linestyle="--", alpha=0.5, label="Parity")
ax.legend()

for bar, pct in zip(bars, top_countries["pct_female"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("figures/female_authorship_by_country.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nUS female first authorship: ", end="")
us = country_gender[country_gender["country_code"] == "US"]
if len(us) > 0:
    print(f"{us.iloc[0]['pct_female']:.1f}%")
    print(f"Paper reference: US male authorship 82.5% (female ~17.5%)")

## 5. Topic × Gender Analysis

Paper finding: Female representation varies by topic from 14.6% (portfolio choice) to 30.0% (credit pricing)

In [ ]:
# Topic × gender analysis
topic_articles = articles_topics[articles_topics["pct_female"].notna()].copy()

topic_gender = topic_articles.groupby("dominant_topic").agg(
    mean_pct_female=("pct_female", "mean"),
    n_articles=("pct_female", "count"),
).reset_index().sort_values("mean_pct_female", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.RdYlBu(np.linspace(0.2, 0.8, len(topic_gender)))
bars = ax.barh(
    [f"Topic {int(t)}" for t in topic_gender["dominant_topic"]],
    topic_gender["mean_pct_female"] * 100,
    color=colors, edgecolor="black", alpha=0.85,
)

# Add value labels
for bar, pct in zip(bars, topic_gender["mean_pct_female"] * 100):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%", va="center", fontsize=9)

ax.set_xlabel("Female Authorship (%)")
ax.set_title("Female Authorship by LDA Topic")
ax.axvline(x=topic_gender["mean_pct_female"].mean() * 100, color="red",
           linestyle="--", alpha=0.7, label="Overall mean")
ax.legend()
plt.tight_layout()
plt.savefig("figures/topic_gender_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nRange: {topic_gender['mean_pct_female'].min()*100:.1f}% - {topic_gender['mean_pct_female'].max()*100:.1f}%")
print(f"Paper reference: 14.6% - 30.0%")

## 6. Sex/Gender Content Analysis

Paper finding: 0.78% of articles have sex/gender content. Split: 51% INS, 32% CSD, 17% GM

In [ ]:
# Sex/gender content prevalence
n_total = len(articles_topics)
n_gender = articles_topics["has_gender_content"].sum()
pct_gender = n_gender / n_total * 100

print(f"Sex/gender content: {n_gender:,} / {n_total:,} ({pct_gender:.2f}%)")
print(f"Paper reference: 433 / 55,210 (0.78%)")

# Category breakdown
gender_arts = articles_topics[articles_topics["has_gender_content"]]
cat_labels = {
    "instrumental": "Instrumental (INS)",
    "sex_differences_catalogue": "Catalogues of Sex Differences (CSD)",
    "gender_mechanisms": "Gender Mechanisms (GM)",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: prevalence over time
yearly_gc = articles_topics.groupby("publication_year").agg(
    n_total=("has_gender_content", "count"),
    n_gender=("has_gender_content", "sum"),
).reset_index()
yearly_gc["pct_gender"] = yearly_gc["n_gender"] / yearly_gc["n_total"] * 100

axes[0].plot(yearly_gc["publication_year"], yearly_gc["pct_gender"],
             color="steelblue", linewidth=2)
axes[0].set_xlabel("Publication Year")
axes[0].set_ylabel("Articles with Sex/Gender Content (%)")
axes[0].set_title("Prevalence of Sex/Gender Content Over Time")
axes[0].grid(True, alpha=0.3)

# Right: category pie chart
if "gender_category" in gender_arts.columns:
    cat_counts = gender_arts["gender_category"].value_counts()
    labels = [cat_labels.get(c, c) for c in cat_counts.index]
    colors_pie = ["#ff9800", "#2196f3", "#4caf50"]
    axes[1].pie(cat_counts, labels=labels, autopct="%1.1f%%", colors=colors_pie[:len(cat_counts)],
                startangle=90, textprops={"fontsize": 10})
    axes[1].set_title("Categories of Sex/Gender Articles")

plt.tight_layout()
plt.savefig("figures/gender_content_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# Gender of authors in sex/gender articles vs. others
gender_with = articles_topics[articles_topics["has_gender_content"] & articles_topics["pct_female"].notna()]
gender_without = articles_topics[~articles_topics["has_gender_content"] & articles_topics["pct_female"].notna()]

pct_f_with = gender_with["pct_female"].mean() * 100
pct_f_without = gender_without["pct_female"].mean() * 100

print(f"\nFemale authorship in sex/gender articles: {pct_f_with:.1f}%")
print(f"Female authorship in other articles: {pct_f_without:.1f}%")
print(f"Difference: {pct_f_with - pct_f_without:.1f} percentage points")
print(f"\nPaper: 33% vs 20% (difference of 12 pp)")

## 7. Spearman Correlation: Female Authorship vs. Sex/Gender Content

Paper finding: ρ = 0.45

In [ ]:
# Journal-level correlation between female authorship and sex/gender content
journal_stats = articles_topics[articles_topics["pct_female"].notna()].groupby("journal_name").agg(
    mean_pct_female=("pct_female", "mean"),
    pct_gender_content=("has_gender_content", "mean"),
    n_articles=("has_gender_content", "count"),
).reset_index()

# Filter to journals with enough articles
journal_stats = journal_stats[journal_stats["n_articles"] >= 20]

# Spearman correlation
rho, p_value = stats.spearmanr(
    journal_stats["mean_pct_female"],
    journal_stats["pct_gender_content"],
)

print(f"Spearman ρ = {rho:.3f} (p = {p_value:.4f})")
print(f"Paper reference: ρ = 0.45")

# Scatter plot
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    journal_stats["mean_pct_female"] * 100,
    journal_stats["pct_gender_content"] * 100,
    s=journal_stats["n_articles"] / 5,  # Size by article count
    alpha=0.6, color="steelblue", edgecolor="black",
)

# Fit line
z = np.polyfit(journal_stats["mean_pct_female"], journal_stats["pct_gender_content"], 1)
p = np.poly1d(z)
x_line = np.linspace(journal_stats["mean_pct_female"].min(), journal_stats["mean_pct_female"].max(), 100)
ax.plot(x_line * 100, p(x_line) * 100, "r--", linewidth=2, alpha=0.7)

ax.set_xlabel("Female Authorship (%)")
ax.set_ylabel("Articles with Sex/Gender Content (%)")
ax.set_title(f"Journal-Level: Female Authorship vs. Sex/Gender Content\n(Spearman ρ = {rho:.3f}, p = {p_value:.4f})")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figures/correlation_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Bayesian Logistic Regression

Model: P(has_gender_content) ~ pct_female + year + journal_rank

The paper used Stan. Here we use `cmdstanpy`. If Stan is not installed, a scipy-based logistic regression fallback is provided.

In [ ]:
# Prepare data for regression
reg_data = articles_topics[
    articles_topics["pct_female"].notna() &
    articles_topics["abdc_rank"].notna()
].copy()

# Encode ABDC rank as numeric (A*=4, A=3, B=2, C=1)
rank_map = {"A*": 4, "A": 3, "B": 2, "C": 1}
reg_data["rank_numeric"] = reg_data["abdc_rank"].map(rank_map)

# Standardize year
reg_data["year_std"] = (reg_data["publication_year"] - reg_data["publication_year"].mean()) / reg_data["publication_year"].std()

# Dependent variable
reg_data["y"] = reg_data["has_gender_content"].astype(int)

print(f"Regression sample: {len(reg_data):,} articles")
print(f"  with gender content: {reg_data['y'].sum():,} ({reg_data['y'].mean()*100:.2f}%)")

# ============================================================
# Try Bayesian approach with cmdstanpy, fallback to frequentist
# ============================================================
try:
    import cmdstanpy
    
    stan_code = """
    data {
      int<lower=0> N;
      array[N] int<lower=0,upper=1> y;
      vector[N] pct_female;
      vector[N] year_std;
      vector[N] rank_numeric;
    }
    parameters {
      real alpha;
      real beta_female;
      real beta_year;
      real beta_rank;
    }
    model {
      alpha ~ normal(0, 5);
      beta_female ~ normal(0, 2);
      beta_year ~ normal(0, 2);
      beta_rank ~ normal(0, 2);
      y ~ bernoulli_logit(alpha + beta_female * pct_female + beta_year * year_std + beta_rank * rank_numeric);
    }
    """
    
    # Write Stan model
    stan_file = "data/processed/gender_content_model.stan"
    with open(stan_file, "w") as f:
        f.write(stan_code)
    
    model = cmdstanpy.CmdStanModel(stan_file=stan_file)
    
    stan_data = {
        "N": len(reg_data),
        "y": reg_data["y"].tolist(),
        "pct_female": reg_data["pct_female"].tolist(),
        "year_std": reg_data["year_std"].tolist(),
        "rank_numeric": reg_data["rank_numeric"].tolist(),
    }
    
    fit = model.sample(data=stan_data, chains=4, iter_sampling=1000, iter_warmup=500, seed=42)
    
    print("\nBayesian Logistic Regression Results:")
    print(fit.summary())
    
    # Plot posteriors
    import arviz as az
    az_data = az.from_cmdstanpy(fit)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, param in enumerate(["beta_female", "beta_year", "beta_rank"]):
        az.plot_posterior(az_data, var_names=[param], ax=axes[i])
        axes[i].set_title(param)
    plt.suptitle("Posterior Distributions - Bayesian Logistic Regression", fontsize=13)
    plt.tight_layout()
    plt.savefig("figures/bayesian_posteriors.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    USE_BAYESIAN = True

except (ImportError, Exception) as e:
    print(f"Stan not available ({e}). Using frequentist logistic regression instead.")
    USE_BAYESIAN = False
    
    from scipy.special import expit
    from scipy.optimize import minimize
    
    # Simple logistic regression via MLE
    X = np.column_stack([
        np.ones(len(reg_data)),
        reg_data["pct_female"].values,
        reg_data["year_std"].values,
        reg_data["rank_numeric"].values,
    ])
    y = reg_data["y"].values
    
    def neg_log_likelihood(beta):
        logits = X @ beta
        logits = np.clip(logits, -500, 500)
        return -np.sum(y * logits - np.log1p(np.exp(logits)))
    
    result = minimize(neg_log_likelihood, x0=np.zeros(4), method="BFGS")
    beta_hat = result.x
    
    param_names = ["intercept", "pct_female", "year_std", "rank_numeric"]
    print("\nFrequentist Logistic Regression Results:")
    print(f"{'Parameter':<20} {'Coefficient':<15} {'Odds Ratio':<15}")
    print("-" * 50)
    for name, coef in zip(param_names, beta_hat):
        print(f"{name:<20} {coef:<15.4f} {np.exp(coef):<15.4f}")
    
    print(f"\nKey finding: pct_female coefficient = {beta_hat[1]:.4f}")
    print(f"Odds ratio = {np.exp(beta_hat[1]):.4f}")
    if beta_hat[1] > 0:
        print("=> Higher female authorship is associated with MORE sex/gender content")
    else:
        print("=> No positive association found (unexpected)")
    print("\nPaper: Positive association between female authorship and gender content")

## 9. Summary of Replication Results

In [ ]:
# ============================================================
# Summary comparison: Our replication vs. Paper
# ============================================================
print("=" * 70)
print("REPLICATION SUMMARY")
print("=" * 70)

# Article count
print(f"\n{'Metric':<45} {'Ours':<15} {'Paper':<15}")
print("-" * 70)
print(f"{'Total articles':<45} {len(articles_all):<15,} {'125,871':<15}")
print(f"{'Articles with abstracts (1988-2020)':<45} {len(articles_topics):<15,} {'55,210':<15}")

# Gender
classified_auth = authorships[authorships["gender"].isin(["male", "female"])]
our_male_pct = (classified_auth["gender"] == "male").mean() * 100
our_female_pct = (classified_auth["gender"] == "female").mean() * 100
print(f"{'Male authorship %':<45} {our_male_pct:<15.1f} {'83.3%':<15}")
print(f"{'Female authorship %':<45} {our_female_pct:<15.1f} {'16.7%':<15}")

# Gender content
print(f"{'Sex/gender content %':<45} {pct_gender:<15.2f} {'0.78%':<15}")

# Correlation
print(f"{'Spearman ρ (female auth vs gender content)':<45} {rho:<15.3f} {'0.45':<15}")

# Topics
print(f"{'Number of LDA topics':<45} {'21':<15} {'21':<15}")

print("\n" + "=" * 70)
print("NOTE: Differences are expected due to different data source (OpenAlex vs.")
print("Dimensions) and gender inference tool (gender-guesser vs. Gender API).")
print("=" * 70)

# List all saved figures
print("\nFigures saved to figures/:")
for f in sorted(os.listdir("figures")):
    print(f"  - {f}")